In [ ]:
# LSTM Model for Temperature Prediction - Khulna
# Predicting April 2024 t2m using March-April data from 2003-2023
# Kaggle-compatible version

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Kaggle paths
KAGGLE_WORKING = '/kaggle/working'
CHECKPOINT_PATH = os.path.join(KAGGLE_WORKING, 'lstm_checkpoint.pth')
DATA_CHECKPOINT_PATH = os.path.join(KAGGLE_WORKING, 'data_checkpoint.pkl')

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
if IS_KAGGLE:
    OUTPUT_DIR = KAGGLE_WORKING
else:
    OUTPUT_DIR = '.'
    
print(f"Running on: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Output directory: {OUTPUT_DIR}")

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


In [ ]:
# Load and preprocess data using xarray (Kaggle compatible)
# Update this path to your Kaggle input path if needed
DATA_PATH = '/kaggle/input/accum-40-yrs-sm-tp-t2m/instant_all.nc'

# Load NetCDF using xarray
ds = xr.open_dataset(DATA_PATH)

print("Dataset variables:", list(ds.data_vars))
print("Dataset dimensions:", dict(ds.dims))

# Extract coordinates
times = pd.to_datetime(ds['valid_time'].values)
lats = ds['latitude'].values
lons = ds['longitude'].values

# Extract variables
t2m = ds['t2m'].values
swvl1 = ds['swvl1'].values

# Get units and convert temperature
t2m_units = ds['t2m'].attrs.get('units', 'K')
if t2m_units == 'K':
    t2m = t2m - 273.15
    print("Temperature converted from Kelvin to Celsius")

# Khulna coordinates
target_lat = 22.8456
target_lon = 89.5403

# Find nearest indices
lat_idx = int(np.abs(lats - target_lat).argmin())
lon_idx = int(np.abs(lons - target_lon).argmin())

print(f"Target: lat={target_lat}, lon={target_lon}")
print(f"Nearest: lat={lats[lat_idx]:.2f}, lon={lons[lon_idx]:.2f}")

# Extract time series for Khulna
temp_series = t2m[:, lat_idx, lon_idx]
sm_series = swvl1[:, lat_idx, lon_idx]

# Handle NaN values
temp_series = np.nan_to_num(temp_series, nan=np.nanmean(temp_series))
sm_series = np.nan_to_num(sm_series, nan=np.nanmean(sm_series))

# Create DataFrame
df = pd.DataFrame({
    'time': times,
    'temperature': temp_series.astype(float),
    'soil_moisture': sm_series.astype(float)
})

df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day

# Close dataset
ds.close()

print(f"\nFull dataset shape: {df.shape}")
print(f"Date range: {df['time'].min()} to {df['time'].max()}")


In [ ]:
# Filter for March and April data (2003-2024)
df_filtered = df[(df['month'].isin([3, 4])) & (df['year'] >= 2003) & (df['year'] <= 2024)].copy()
df_filtered = df_filtered.dropna()

# Sort by time
df_filtered = df_filtered.sort_values('time').reset_index(drop=True)

print(f"Filtered dataset shape: {df_filtered.shape}")
print(f"Years covered: {df_filtered['year'].min()} to {df_filtered['year'].max()}")
print(f"\nData points per year:")
print(df_filtered.groupby('year').size())

# Separate training data (2003-2023 + March 2024) and test data (April 2024)
# Training: March-April 2003-2023 AND March 2024
df_train = df_filtered[
    (df_filtered['year'] <= 2023) | 
    ((df_filtered['year'] == 2024) & (df_filtered['month'] == 3))
].copy()

# Test: April 2024 only
df_test = df_filtered[(df_filtered['year'] == 2024) & (df_filtered['month'] == 4)].copy()

print(f"\nTraining data: {df_train.shape[0]} samples")
print(f"  - 2003-2023 (March-April): {df_train[df_train['year'] <= 2023].shape[0]} samples")
print(f"  - March 2024: {df_train[(df_train['year'] == 2024) & (df_train['month'] == 3)].shape[0]} samples")
print(f"Test data (April 2024): {df_test.shape[0]} samples")

# Save data checkpoint for session recovery
data_checkpoint = {
    'df_filtered': df_filtered,
    'df_train': df_train,
    'df_test': df_test,
    'target_lat': target_lat,
    'target_lon': target_lon
}
with open(os.path.join(OUTPUT_DIR, 'data_checkpoint.pkl'), 'wb') as f:
    pickle.dump(data_checkpoint, f)
print(f"\nData checkpoint saved to {OUTPUT_DIR}/data_checkpoint.pkl")


In [ ]:
# Prepare features and target
features = ['temperature', 'soil_moisture']
target = 'temperature'

# Scale the data
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit scalers on training data only
X_train_scaled = scaler_X.fit_transform(df_train[features].values)
y_train_scaled = scaler_y.fit_transform(df_train[[target]].values)

# Transform test data
X_test_scaled = scaler_X.transform(df_test[features].values)
y_test_scaled = scaler_y.transform(df_test[[target]].values)

print(f"Scaled training features shape: {X_train_scaled.shape}")
print(f"Scaled test features shape: {X_test_scaled.shape}")

# Save scalers checkpoint
scalers_checkpoint = {
    'scaler_X': scaler_X,
    'scaler_y': scaler_y,
    'features': features,
    'target': target
}
with open(os.path.join(OUTPUT_DIR, 'scalers_checkpoint.pkl'), 'wb') as f:
    pickle.dump(scalers_checkpoint, f)
print(f"Scalers checkpoint saved to {OUTPUT_DIR}/scalers_checkpoint.pkl")


In [ ]:
# Create sequences for LSTM
def create_sequences(X, y, seq_length):
    """Create sequences for LSTM training"""
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

# Sequence length (number of previous time steps to use)
SEQ_LENGTH = 30

# Create sequences from training data
X_seq, y_seq = create_sequences(X_train_scaled, y_train_scaled, SEQ_LENGTH)

print(f"Sequence shape: X={X_seq.shape}, y={y_seq.shape}")

# Split into train and validation sets (80-20)
split_idx = int(len(X_seq) * 0.8)
X_train_seq = X_seq[:split_idx]
y_train_seq = y_seq[:split_idx]
X_val_seq = X_seq[split_idx:]
y_val_seq = y_seq[split_idx:]

print(f"\nTraining sequences: {X_train_seq.shape[0]}")
print(f"Validation sequences: {X_val_seq.shape[0]}")


In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_seq).to(device)
y_train_tensor = torch.FloatTensor(y_train_seq).to(device)
X_val_tensor = torch.FloatTensor(X_val_seq).to(device)
y_val_tensor = torch.FloatTensor(y_val_seq).to(device)

# Create DataLoaders
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


In [ ]:
# Define LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(LSTMModel, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size // 2, output_size)
        
    def forward(self, x):
        # Initialize hidden state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # LSTM forward pass
        out, _ = self.lstm(x, (h0, c0))
        
        # Take the output from the last time step
        out = out[:, -1, :]
        
        # Fully connected layers
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

# Model hyperparameters
INPUT_SIZE = len(features)  # Number of features
HIDDEN_SIZE = 64
NUM_LAYERS = 2
OUTPUT_SIZE = 1
DROPOUT = 0.2

# Initialize model
model = LSTMModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE, DROPOUT).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Training configuration
EPOCHS = 100
LEARNING_RATE = 0.001
CHECKPOINT_INTERVAL = 5  # Save checkpoint every N epochs

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5, verbose=True)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_rmse': [],
    'val_rmse': [],
    'train_mae': [],
    'val_mae': [],
    'lr': []
}

# Early stopping
best_val_loss = float('inf')
patience = 20
patience_counter = 0
best_model_state = None
start_epoch = 0

# Check for existing checkpoint to resume training
checkpoint_file = os.path.join(OUTPUT_DIR, 'lstm_checkpoint.pth')
if os.path.exists(checkpoint_file):
    print("Loading checkpoint to resume training...")
    checkpoint = torch.load(checkpoint_file, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_val_loss = checkpoint['best_val_loss']
    best_model_state = checkpoint.get('best_model_state', None)
    print(f"Resumed from epoch {start_epoch}, best val loss: {best_val_loss:.6f}")
else:
    print("No checkpoint found, starting fresh training...")

print("\n" + "="*100)
print(f"{'Epoch':^6} | {'Train Loss':^12} | {'Val Loss':^12} | {'Train RMSE':^12} | {'Val RMSE':^12} | {'Train MAE':^12} | {'Val MAE':^12} | {'LR':^10}")
print("="*100)


In [ ]:
# Training loop with checkpoint saving
for epoch in range(start_epoch, EPOCHS):
    # Training phase
    model.train()
    train_losses = []
    train_preds = []
    train_targets = []
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_losses.append(loss.item())
        train_preds.extend(y_pred.detach().cpu().numpy())
        train_targets.extend(y_batch.detach().cpu().numpy())
    
    # Validation phase
    model.eval()
    val_losses = []
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            
            val_losses.append(loss.item())
            val_preds.extend(y_pred.cpu().numpy())
            val_targets.extend(y_batch.cpu().numpy())
    
    # Calculate metrics
    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    
    # Inverse transform for actual metrics
    train_preds_inv = scaler_y.inverse_transform(np.array(train_preds).reshape(-1, 1)).flatten()
    train_targets_inv = scaler_y.inverse_transform(np.array(train_targets).reshape(-1, 1)).flatten()
    val_preds_inv = scaler_y.inverse_transform(np.array(val_preds).reshape(-1, 1)).flatten()
    val_targets_inv = scaler_y.inverse_transform(np.array(val_targets).reshape(-1, 1)).flatten()
    
    train_rmse = np.sqrt(mean_squared_error(train_targets_inv, train_preds_inv))
    val_rmse = np.sqrt(mean_squared_error(val_targets_inv, val_preds_inv))
    train_mae = mean_absolute_error(train_targets_inv, train_preds_inv)
    val_mae = mean_absolute_error(val_targets_inv, val_preds_inv)
    
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_rmse'].append(train_rmse)
    history['val_rmse'].append(val_rmse)
    history['train_mae'].append(train_mae)
    history['val_mae'].append(val_mae)
    history['lr'].append(current_lr)
    
    # Print epoch results
    print(f"{epoch+1:^6} | {train_loss:^12.6f} | {val_loss:^12.6f} | {train_rmse:^12.4f} | {val_rmse:^12.4f} | {train_mae:^12.4f} | {val_mae:^12.4f} | {current_lr:^10.6f}")
    
    # Learning rate scheduler
    scheduler.step(val_loss)
    
    # Early stopping check and best model saving
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        # Save best model checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'best_model_state': best_model_state,
            'history': history
        }, os.path.join(OUTPUT_DIR, 'lstm_best_model.pth'))
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break
    
    # Save periodic checkpoint every CHECKPOINT_INTERVAL epochs
    if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
        checkpoint_path = os.path.join(OUTPUT_DIR, 'lstm_checkpoint.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'best_model_state': best_model_state,
            'history': history
        }, checkpoint_path)
        print(f"  >> Checkpoint saved at epoch {epoch+1}")

print("="*100)
print("Training completed!")

# Save final checkpoint
final_checkpoint_path = os.path.join(OUTPUT_DIR, 'lstm_final_checkpoint.pth')
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_val_loss': best_val_loss,
    'best_model_state': best_model_state,
    'history': history
}, final_checkpoint_path)
print(f"Final checkpoint saved to {final_checkpoint_path}")

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Loaded best model with validation loss: {best_val_loss:.6f}")


In [ ]:
# Plot Training History
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss plot
ax1 = axes[0, 0]
ax1.plot(history['train_loss'], label='Training Loss', color='blue', linewidth=2)
ax1.plot(history['val_loss'], label='Validation Loss', color='red', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# RMSE plot
ax2 = axes[0, 1]
ax2.plot(history['train_rmse'], label='Training RMSE', color='blue', linewidth=2)
ax2.plot(history['val_rmse'], label='Validation RMSE', color='red', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('RMSE (°C)')
ax2.set_title('Training and Validation RMSE')
ax2.legend()
ax2.grid(True, alpha=0.3)

# MAE plot
ax3 = axes[1, 0]
ax3.plot(history['train_mae'], label='Training MAE', color='blue', linewidth=2)
ax3.plot(history['val_mae'], label='Validation MAE', color='red', linewidth=2)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('MAE (°C)')
ax3.set_title('Training and Validation MAE')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Learning rate plot
ax4 = axes[1, 1]
ax4.plot(history['lr'], color='green', linewidth=2)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Learning Rate')
ax4.set_title('Learning Rate Schedule')
ax4.grid(True, alpha=0.3)
ax4.set_yscale('log')

plt.suptitle('LSTM Training History - Khulna Temperature Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lstm_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Training Metrics:")
print(f"  RMSE: {history['train_rmse'][-1]:.4f}°C")
print(f"  MAE:  {history['train_mae'][-1]:.4f}°C")
print(f"\nFinal Validation Metrics:")
print(f"  RMSE: {history['val_rmse'][-1]:.4f}°C")
print(f"  MAE:  {history['val_mae'][-1]:.4f}°C")


In [ ]:
# Prepare data for April 2024 prediction
# We need the last SEQ_LENGTH values before April 2024 to predict each day

# Get March 2024 and earlier data for context
df_context = df_filtered[df_filtered['time'] < df_test['time'].min()].copy()
df_context = df_context.sort_values('time').reset_index(drop=True)

# Get the last SEQ_LENGTH points before April 2024
context_features = scaler_X.transform(df_context[features].values)

print(f"Context data points: {len(context_features)}")
print(f"Test data points (April 2024): {len(df_test)}")

# Prepare sequences for April 2024 prediction
# We'll use a sliding window approach
all_data = np.vstack([context_features, X_test_scaled])
all_actual = np.concatenate([
    scaler_y.transform(df_context[[target]].values).flatten(),
    y_test_scaled.flatten()
])

# Create prediction sequences for April 2024
test_start_idx = len(context_features)
predictions_april = []
actuals_april = []

model.eval()
with torch.no_grad():
    for i in range(len(df_test)):
        # Get sequence ending at this point
        seq_end_idx = test_start_idx + i
        seq_start_idx = seq_end_idx - SEQ_LENGTH
        
        if seq_start_idx >= 0:
            X_seq = all_data[seq_start_idx:seq_end_idx]
            X_tensor = torch.FloatTensor(X_seq).unsqueeze(0).to(device)
            
            pred = model(X_tensor)
            predictions_april.append(pred.cpu().numpy().flatten()[0])
            actuals_april.append(all_actual[seq_end_idx])

# Inverse transform predictions
predictions_april = scaler_y.inverse_transform(np.array(predictions_april).reshape(-1, 1)).flatten()
actuals_april = scaler_y.inverse_transform(np.array(actuals_april).reshape(-1, 1)).flatten()

print(f"\nPredictions made: {len(predictions_april)}")
print(f"Actual values: {len(actuals_april)}")


In [ ]:
# Calculate April 2024 prediction metrics
april_rmse = np.sqrt(mean_squared_error(actuals_april, predictions_april))
april_mae = mean_absolute_error(actuals_april, predictions_april)
april_r2 = r2_score(actuals_april, predictions_april)
april_mape = np.mean(np.abs((actuals_april - predictions_april) / actuals_april)) * 100

print("="*60)
print("April 2024 Prediction Performance")
print("="*60)
print(f"RMSE:  {april_rmse:.4f}°C")
print(f"MAE:   {april_mae:.4f}°C")
print(f"R²:    {april_r2:.4f}")
print(f"MAPE:  {april_mape:.2f}%")
print("="*60)

# Create prediction DataFrame
pred_dates = df_test['time'].values[:len(predictions_april)]
pred_df = pd.DataFrame({
    'Date': pred_dates,
    'Actual': actuals_april,
    'Predicted': predictions_april,
    'Error': predictions_april - actuals_april,
    'Abs_Error': np.abs(predictions_april - actuals_april)
})

print("\nApril 2024 Predictions Summary:")
print(pred_df.describe())


In [ ]:
# Comprehensive Visualization - Figure 1: Time Series Comparison
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Actual vs Predicted Time Series
ax1 = axes[0]
x_dates = range(len(predictions_april))
ax1.plot(x_dates, actuals_april, 'b-', label='Actual Temperature', linewidth=2, marker='o', markersize=4)
ax1.plot(x_dates, predictions_april, 'r--', label='Predicted Temperature', linewidth=2, marker='s', markersize=4)
ax1.fill_between(x_dates, actuals_april, predictions_april, alpha=0.3, color='gray')
ax1.set_xlabel('Time Steps (April 2024)')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Actual vs Predicted Temperature - April 2024', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction Error Over Time
ax2 = axes[1]
errors = predictions_april - actuals_april
colors = ['green' if e >= 0 else 'red' for e in errors]
ax2.bar(x_dates, errors, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.axhline(y=np.mean(errors), color='blue', linestyle='--', linewidth=2, label=f'Mean Error: {np.mean(errors):.2f}°C')
ax2.axhline(y=np.mean(errors) + np.std(errors), color='orange', linestyle=':', linewidth=1.5, label=f'±1 Std: {np.std(errors):.2f}°C')
ax2.axhline(y=np.mean(errors) - np.std(errors), color='orange', linestyle=':', linewidth=1.5)
ax2.set_xlabel('Time Steps (April 2024)')
ax2.set_ylabel('Prediction Error (°C)')
ax2.set_title('Prediction Error Over Time', fontsize=12, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Plot 3: Cumulative Absolute Error
ax3 = axes[2]
cumulative_error = np.cumsum(np.abs(errors))
ax3.plot(x_dates, cumulative_error, 'purple', linewidth=2, marker='d', markersize=4)
ax3.fill_between(x_dates, 0, cumulative_error, alpha=0.3, color='purple')
ax3.set_xlabel('Time Steps (April 2024)')
ax3.set_ylabel('Cumulative Absolute Error (°C)')
ax3.set_title('Cumulative Absolute Error', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

plt.suptitle('LSTM Temperature Prediction Analysis - Khulna, April 2024', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lstm_prediction_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Comprehensive Visualization - Figure 2: Statistical Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Scatter Plot - Actual vs Predicted
ax1 = axes[0, 0]
ax1.scatter(actuals_april, predictions_april, alpha=0.6, c='blue', edgecolors='black', s=60)
min_val = min(actuals_april.min(), predictions_april.min())
max_val = max(actuals_april.max(), predictions_april.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

# Add regression line
z = np.polyfit(actuals_april, predictions_april, 1)
p = np.poly1d(z)
ax1.plot(actuals_april, p(actuals_april), 'g-', linewidth=2, label=f'Regression (slope={z[0]:.3f})')

ax1.set_xlabel('Actual Temperature (°C)')
ax1.set_ylabel('Predicted Temperature (°C)')
ax1.set_title(f'Actual vs Predicted (R² = {april_r2:.4f})', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Error Distribution Histogram
ax2 = axes[0, 1]
n, bins, patches = ax2.hist(errors, bins=20, color='steelblue', edgecolor='black', alpha=0.7, density=True)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax2.axvline(x=np.mean(errors), color='green', linestyle='-', linewidth=2, label=f'Mean: {np.mean(errors):.2f}°C')

# Add normal distribution curve
from scipy import stats
mu, std = np.mean(errors), np.std(errors)
x_norm = np.linspace(errors.min(), errors.max(), 100)
ax2.plot(x_norm, stats.norm.pdf(x_norm, mu, std), 'orange', linewidth=2, label='Normal Distribution')

ax2.set_xlabel('Prediction Error (°C)')
ax2.set_ylabel('Density')
ax2.set_title('Error Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Box Plot of Errors
ax3 = axes[1, 0]
bp = ax3.boxplot([errors], labels=['Prediction Error'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][0].set_alpha(0.7)
ax3.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax3.set_ylabel('Error (°C)')
ax3.set_title('Error Distribution Box Plot', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Add statistics annotation
stats_text = f'Mean: {np.mean(errors):.2f}°C\nMedian: {np.median(errors):.2f}°C\nStd: {np.std(errors):.2f}°C\nIQR: {np.percentile(errors, 75) - np.percentile(errors, 25):.2f}°C'
ax3.text(1.3, np.mean(errors), stats_text, fontsize=10, verticalalignment='center',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Residual Plot
ax4 = axes[1, 1]
ax4.scatter(predictions_april, errors, alpha=0.6, c='purple', edgecolors='black', s=60)
ax4.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax4.axhline(y=np.mean(errors) + 2*np.std(errors), color='orange', linestyle=':', linewidth=1.5, label='±2 Std')
ax4.axhline(y=np.mean(errors) - 2*np.std(errors), color='orange', linestyle=':', linewidth=1.5)
ax4.set_xlabel('Predicted Temperature (°C)')
ax4.set_ylabel('Residual (°C)')
ax4.set_title('Residual Plot', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.suptitle('LSTM Prediction Statistical Analysis - Khulna, April 2024', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lstm_prediction_statistics.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Comprehensive Visualization - Figure 3: Historical Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Get historical April data for comparison
df_april_hist = df_train[df_train['month'] == 4].copy()

# Plot 1: Historical April Temperatures by Year
ax1 = axes[0, 0]
years = df_april_hist['year'].unique()
for year in years[-5:]:  # Last 5 years
    year_data = df_april_hist[df_april_hist['year'] == year]['temperature'].values
    ax1.plot(range(len(year_data)), year_data, alpha=0.5, label=f'{year}')

# Add 2024 actual and predicted
ax1.plot(range(len(actuals_april)), actuals_april, 'b-', linewidth=2.5, label='2024 Actual')
ax1.plot(range(len(predictions_april)), predictions_april, 'r--', linewidth=2.5, label='2024 Predicted')
ax1.set_xlabel('Day of April')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Historical April Temperature Comparison', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: April Mean Temperature by Year
ax2 = axes[0, 1]
yearly_means = df_april_hist.groupby('year')['temperature'].mean()
ax2.bar(yearly_means.index, yearly_means.values, color='steelblue', alpha=0.7, edgecolor='black')
ax2.axhline(y=np.mean(actuals_april), color='blue', linestyle='-', linewidth=2, label=f'2024 Actual Mean: {np.mean(actuals_april):.2f}°C')
ax2.axhline(y=np.mean(predictions_april), color='red', linestyle='--', linewidth=2, label=f'2024 Predicted Mean: {np.mean(predictions_april):.2f}°C')
ax2.axhline(y=yearly_means.mean(), color='green', linestyle=':', linewidth=2, label=f'Historical Mean: {yearly_means.mean():.2f}°C')
ax2.set_xlabel('Year')
ax2.set_ylabel('Mean Temperature (°C)')
ax2.set_title('April Mean Temperature by Year', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

# Plot 3: Temperature Distribution Comparison
ax3 = axes[1, 0]
ax3.hist(df_april_hist['temperature'].values, bins=30, alpha=0.5, label='Historical April', color='gray', density=True)
ax3.hist(actuals_april, bins=15, alpha=0.7, label='2024 Actual', color='blue', density=True)
ax3.hist(predictions_april, bins=15, alpha=0.7, label='2024 Predicted', color='red', density=True)
ax3.set_xlabel('Temperature (°C)')
ax3.set_ylabel('Density')
ax3.set_title('Temperature Distribution Comparison', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Model Performance Summary
ax4 = axes[1, 1]
metrics = ['RMSE\n(°C)', 'MAE\n(°C)', 'R²', 'MAPE\n(%)']
values = [april_rmse, april_mae, april_r2, april_mape]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
bars = ax4.bar(metrics, values, color=colors, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax4.set_ylabel('Value')
ax4.set_title('April 2024 Prediction Performance Metrics', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('LSTM Model Performance Summary - Khulna Temperature Prediction', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lstm_performance_summary.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Final Summary and Save Results
print("="*70)
print("LSTM Temperature Prediction Model - Final Summary")
print("="*70)
print(f"\nLocation: Khulna (lat={target_lat}, lon={target_lon})")
print(f"Training Period: March-April 2003-2023 + March 2024")
print(f"Test Period: April 2024")
print(f"\nModel Architecture:")
print(f"  - LSTM Layers: {NUM_LAYERS}")
print(f"  - Hidden Size: {HIDDEN_SIZE}")
print(f"  - Sequence Length: {SEQ_LENGTH}")
print(f"  - Dropout: {DROPOUT}")
print(f"  - Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTraining Configuration:")
print(f"  - Epochs: {len(history['train_loss'])}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Initial Learning Rate: {LEARNING_RATE}")
print(f"  - Optimizer: Adam")
print(f"\nApril 2024 Prediction Performance:")
print(f"  - RMSE: {april_rmse:.4f}°C")
print(f"  - MAE: {april_mae:.4f}°C")
print(f"  - R²: {april_r2:.4f}")
print(f"  - MAPE: {april_mape:.2f}%")
print(f"\nTemperature Statistics:")
print(f"  - Actual Mean: {np.mean(actuals_april):.2f}°C")
print(f"  - Predicted Mean: {np.mean(predictions_april):.2f}°C")
print(f"  - Actual Range: {actuals_april.min():.2f}°C to {actuals_april.max():.2f}°C")
print(f"  - Predicted Range: {predictions_april.min():.2f}°C to {predictions_april.max():.2f}°C")
print("="*70)

# Save predictions to CSV
pred_csv_path = os.path.join(OUTPUT_DIR, 'lstm_predictions_april2024.csv')
pred_df.to_csv(pred_csv_path, index=False)
print(f"\nPredictions saved to '{pred_csv_path}'")

# Save model
model_path = os.path.join(OUTPUT_DIR, 'lstm_model_khulna.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scaler_X': scaler_X,
    'scaler_y': scaler_y,
    'history': history,
    'config': {
        'input_size': INPUT_SIZE,
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LAYERS,
        'output_size': OUTPUT_SIZE,
        'seq_length': SEQ_LENGTH,
        'dropout': DROPOUT
    }
}, model_path)
print(f"Model saved to '{model_path}'")

# List all saved files
print(f"\n{'='*70}")
print("All saved files in output directory:")
print("="*70)
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {f}: {size_kb:.2f} KB")

# Display prediction table
print("\n" + "="*70)
print("April 2024 Daily Predictions")
print("="*70)
print(pred_df.to_string(index=False))
